In [1]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("austinreese/craigslist-carstrucks-data")
print("Path to dataset files:", path)

cars = pd.read_csv(f"{path}/vehicles.csv")
print(cars.shape)
print(cars.columns.tolist())
cars.head()

Path to dataset files: /Users/abaid/.cache/kagglehub/datasets/austinreese/craigslist-carstrucks-data/versions/10
(426880, 26)
['id', 'url', 'region', 'region_url', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'image_url', 'description', 'county', 'state', 'lat', 'long', 'posting_date']


,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


In [2]:
print(cars["description"].isna().sum(), "missing descriptions")
print(cars["description"].str.split().str.len().describe())
cars[["type", "fuel", "transmission", "drive", "condition"]].nunique()

70 missing descriptions
count    426810.000000
mean        433.005904
std         419.547700
min           1.000000
25%          92.000000
50%         357.000000
75%         654.000000
max        4777.000000
Name: description, dtype: float64


type            13
fuel             5
transmission     3
drive            3
condition        6
dtype: int64

In [3]:
# actual category values, not just counts
for col in ["type", "fuel", "transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars[col].value_counts(dropna=False))
    print()

--- type ---
type
NaN            92858
sedan          87056
SUV            77284
pickup         43510
truck          35279
other          22110
coupe          19204
hatchback      16598
wagon          10751
van             8548
convertible     7731
mini-van        4825
offroad          609
bus              517
Name: count, dtype: int64

--- fuel ---
fuel
gas         356209
other        30728
diesel       30062
hybrid        5170
NaN           3013
electric      1698
Name: count, dtype: int64

--- transmission ---
transmission
automatic    336524
other         62682
manual        25118
NaN            2556
Name: count, dtype: int64

--- drive ---
drive
4wd    131904
NaN    130567
fwd    105517
rwd     58892
Name: count, dtype: int64

--- condition ---
condition
NaN          174104
good         121456
excellent    101467
like new      21178
fair           6769
new            1305
salvage         601
Name: count, dtype: int64



In [4]:
# this dataset is notorious for duplicate/reposted listings across regions
print("duplicate descriptions:", cars["description"].duplicated().sum())
print("duplicate VIN (non-null only):", cars.loc[cars["VIN"].notna(), "VIN"].duplicated().sum())

duplicate descriptions: 65968
duplicate VIN (non-null only): 147574


In [5]:
# eyeball a few short ones and a few very long ones to judge signal vs. boilerplate
word_counts = cars["description"].str.split().str.len()

print("--- SHORT (5-15 words) ---")
for d in cars.loc[word_counts.between(5, 15), "description"].sample(3, random_state=1):
    print(d, "\n")

print("--- LONG (2000+ words) ---")
for d in cars.loc[word_counts >= 2000, "description"].sample(2, random_state=1):
    print(d[:800], "...\n")

--- SHORT (5-15 words) ---
Very beautiful car for sale 

LOADED! NAVIGATION! MOONROOF! RARE INTERIOR! 4WD! WARRANTY! NEW INSPECTION! 

LT white 4cyl auto 4door fully loaded stereo cd alloys runs great 134k call 518-573-6414 

--- LONG (2000+ words) ---
2017 *** BMW 5 Series 530i 4dr Sedan Sedan ***    CALL/TEXT JASMINE (916) 586-6111Call ☏ (916) 586−6111  Sacramento Luxury Motors 7801 Fair Oaks Blvd, Carmichael, CA 95608Copy & Paste the URL belowto view more information!http://sacramentoluxurymotors.v12soft.com/cars/13451361    			Year : 2017				Make : BMW				Model : 5 Series				Trim : 530i 4dr Sedan				   Mileage : 46,345 miles				Transmission : Automatic				Exterior Color : White				Interior Color : Black				Series : 530i 4dr Sedan Sedan				Drivetrain : RWD				Condition : Excellent				VIN : WBAJA5C30HG895266				Stock ID : 14015				Engine : 2.0L I4	   	Description of this BMW 5 Series 530i 4dr Sedan 	 	WWW.SACRAMENTOLUXURYMOTORS.COM Call us @ 916 944 1480 Text JASMIN @ 916 586 6111 78

In [6]:
cars[["price", "odometer", "year"]].describe()

,price,odometer,year
count,4.268800e+05,4.224800e+05,425675.000000
mean,7.519903e+04,9.804333e+04,2011.235191
std,1.218228e+07,2.138815e+05,9.452120
min,0.000000e+00,0.000000e+00,1900.000000
25%,5.900000e+03,3.770400e+04,2008.000000
50%,1.395000e+04,8.554800e+04,2013.000000
75%,2.648575e+04,1.335425e+05,2017.000000
max,3.736929e+09,1.000000e+07,2022.000000


In [7]:
import re

# drop missing descriptions
cars = cars[cars["description"].notna()].copy()

# drop exact duplicate ads (dealer reposts across regions)
cars = cars.drop_duplicates(subset="description", keep="first")

# word-count filter: cuts near-empty ads AND spam-length boilerplate
word_count = cars["description"].str.split().str.len()
cars = cars[word_count.between(15, 200)]

# realistic value ranges — drops scam/typo entries at the extremes
cars = cars[cars["price"].between(500, 100_000)]
cars = cars[cars["odometer"].between(500, 300_000)]
cars = cars[cars["year"].between(1990, 2022)]

print(cars.shape)

(108390, 26)


In [8]:
def clean_description(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)                      # URLs
    text = re.sub(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', ' ', text)    # phone numbers
    text = re.sub(r'[☏✅🚘📞]+', ' ', text)                              # symbols seen in your samples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cars["description_clean"] = cars["description"].apply(clean_description)

# sanity check after cleaning
print(cars.shape)
print(cars["description_clean"].str.split().str.len().describe())
for d in cars["description_clean"].sample(5, random_state=1):
    print(d, "\n")

(108390, 27)
count    108390.000000
mean         76.194898
std          48.252887
min          11.000000
25%          38.000000
50%          62.000000
75%         106.000000
max         201.000000
Name: description_clean, dtype: float64
2003 Honda Accord with 166k everything works except radio,runs great ,fast car black leather , sunroof ,remote entry , nice 18 inch wheels show contact info 

88,261 original mile hard to find 2008 Ford F-350 XLT 4wd with the 84” cab to axle. It is ready for a 10’-12’ flatbed or service body. It was a locally owned and is ready for your use. Please call/text 

Selling our 2014 Dodge Grand Caravan. Excellent condition. Looking to downsize to smaller vehicle. Call or text ninezerosix 360-twofoureighttwo 

For Sale: 2014 Ford Police Interceptor Sedan 3.7L AWD. Good condition, Heated seats, no dents, no rust. Drives nice, no issues, 1,000 miles on new firestone firehawk GT pursuit tires.Replaced engine with 3.7L from a 2013 Explorer that has 65,000 miles on

In [9]:
# how much missingness survived the filters?
print(cars[["type","fuel","transmission","drive","condition"]].isna().sum())
print("remaining duplicate VINs:", cars.loc[cars["VIN"].notna(), "VIN"].duplicated().sum())

type            34568
fuel              314
transmission      109
drive           31394
condition       28358
dtype: int64
remaining duplicate VINs: 4056


In [11]:
cars_sampled = (
    cars.groupby("type")  # Let Pandas keep the group key in the index
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(level=0)  # Brings 'type' out of the index back to a column
    .reset_index(drop=True) # Drops the old row numeric index
)

In [12]:
import pandas as pd
print(pd.__version__)

3.0.5


In [13]:
import numpy as np

sampled_idx = (
    cars.groupby("type")
    .apply(lambda g: g.sample(n=min(len(g), 1000), random_state=42).index)
)
all_idx = np.concatenate(sampled_idx.values)

cars_sampled = cars.loc[all_idx].reset_index(drop=True)

print(cars_sampled.shape)
print(cars_sampled["type"].value_counts())

(12429, 27)
type
SUV            1000
convertible    1000
coupe          1000
hatchback      1000
mini-van       1000
pickup         1000
sedan          1000
truck          1000
unknown        1000
van            1000
wagon          1000
other           914
offroad         311
bus             204
Name: count, dtype: int64


In [14]:
cars_sampled["tagged_description"] = cars_sampled["id"].astype(str) + " " + cars_sampled["description_clean"]

cars_sampled.to_csv("cars_cleaned.csv", index=False)
print(cars_sampled.shape)
cars_sampled[["id", "type", "manufacturer", "model", "price", "tagged_description"]].head()

(12429, 28)


,id,type,manufacturer,model,price,tagged_description
0,7314068693,SUV,jeep,wrangler,14699,7314068693 2008 Jeep Wrangler X Sport Utility ...
1,7311964042,SUV,NaN,HUMMER H2,22997,7311964042 2003 HUMMER H2 Adventure Series 4dr...
2,7305257417,SUV,chevrolet,trailblazer ls,5500,7305257417 Asking $5500 obo all offers welcome...
3,7311037499,SUV,honda,pilot ex-l,10995,"7311037499 2013 Honda Pilot EX-L, 3.5L V6, Aut..."
4,7315419164,SUV,cadillac,escalade,30777,7315419164 2013 Cadillac Escalade Premium 4dr ...


In [15]:
for col in ["type", "fuel", "transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars_sampled[col].value_counts(dropna=False))
    print()

print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["price"].describe())

--- type ---
type
SUV            1000
convertible    1000
coupe          1000
hatchback      1000
mini-van       1000
pickup         1000
sedan          1000
truck          1000
unknown        1000
van            1000
wagon          1000
other           914
offroad         311
bus             204
Name: count, dtype: int64

--- fuel ---
fuel
gas         11260
diesel        806
hybrid        175
other         113
NaN            38
electric       37
Name: count, dtype: int64

--- transmission ---
transmission
automatic    10807
manual        1418
other          187
NaN             17
Name: count, dtype: int64

--- drive ---
drive
fwd    3986
4wd    3423
rwd    2771
NaN    2249
Name: count, dtype: int64

--- condition ---
condition
excellent    4400
good         3840
NaN          2178
like new     1309
fair          606
salvage        48
new            48
Name: count, dtype: int64

unique manufacturers: 38
count    12429.000000
mean     12688.527878
std      11306.933675
min        500.000

In [16]:
for col in ["transmission", "drive", "condition"]:
    print(f"--- {col} ---")
    print(cars_sampled[col].value_counts(dropna=False))
    print()

print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["manufacturer"].value_counts(dropna=False).head(15))

--- transmission ---
transmission
automatic    10807
manual        1418
other          187
NaN             17
Name: count, dtype: int64

--- drive ---
drive
fwd    3986
4wd    3423
rwd    2771
NaN    2249
Name: count, dtype: int64

--- condition ---
condition
excellent    4400
good         3840
NaN          2178
like new     1309
fair          606
salvage        48
new            48
Name: count, dtype: int64

unique manufacturers: 38
manufacturer
ford             2338
chevrolet        1427
toyota           1058
honda             780
NaN               698
dodge             557
subaru            526
jeep              524
nissan            482
ram               380
bmw               366
chrysler          362
gmc               323
mercedes-benz     312
volkswagen        272
Name: count, dtype: int64


In [17]:
print("unique manufacturers:", cars_sampled["manufacturer"].nunique())
print(cars_sampled["manufacturer"].value_counts(dropna=False).head(15))

unique manufacturers: 38
manufacturer
ford             2338
chevrolet        1427
toyota           1058
honda             780
NaN               698
dodge             557
subaru            526
jeep              524
nissan            482
ram               380
bmw               366
chrysler          362
gmc               323
mercedes-benz     312
volkswagen        272
Name: count, dtype: int64


In [18]:
import pandas as pd
cars = pd.read_csv("cars_cleaned.csv")

import torch
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

vibe_labels = ["family car", "sports car", "luxury car", "eco-friendly car", "off-road vehicle", "daily commuter car"]

Using device: mps


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [19]:
import time

sample = cars["description_clean"].sample(20, random_state=1).tolist()

start = time.time()
results = classifier(sample, vibe_labels, multi_label=True)
elapsed = time.time() - start

print(f"{elapsed:.1f}s for 20 rows -> ~{elapsed/20*12429/60:.1f} min estimated for full dataset")

# eyeball a couple to judge label quality
for text, res in zip(sample[:3], results[:3]):
    print(text[:150])
    print(dict(zip(res["labels"], [round(s,2) for s in res["scores"]])))
    print()

9.2s for 20 rows -> ~95.7 min estimated for full dataset
I have a 2015 Chevy malibu with only 14,000 miles on it. It has an rebuilt title, has cold A/C, the interior is in fair conditions, everything is up t
{'family car': 0.5, 'daily commuter car': 0.13, 'eco-friendly car': 0.05, 'luxury car': 0.04, 'off-road vehicle': 0.0, 'sports car': 0.0}

I have a 2002 Buick Lesabre with 176838 miles on it. It runs and am willing to take $1000 obo in the form of cash/Paypal/Venmo. It has newer tires on 
{'family car': 0.5, 'off-road vehicle': 0.48, 'sports car': 0.37, 'luxury car': 0.37, 'daily commuter car': 0.19, 'eco-friendly car': 0.04}

BMW Series 3 328i year 2008 sunroof nice and clean interior and exterior no damadge price 7900.00
{'luxury car': 1.0, 'eco-friendly car': 0.91, 'family car': 0.91, 'daily commuter car': 0.53, 'sports car': 0.5, 'off-road vehicle': 0.06}



In [20]:
import torch
print("MPS available:", torch.backends.mps.is_available())

MPS available: True


In [21]:
vibe_labels = [
    "a family-friendly car with practical space",
    "a sporty high-performance car",
    "a luxury car with premium features",
    "a fuel-efficient eco-friendly car",
    "an off-road or rugged utility vehicle",
    "a basic reliable daily commuter car",
]

hypothesis_template = "This vehicle listing describes {}."

import time
sample = cars["description_clean"].sample(20, random_state=1).tolist()

start = time.time()
results = classifier(
    sample, vibe_labels,
    multi_label=True,
    hypothesis_template=hypothesis_template,
    batch_size=8,
)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 20 rows -> ~{elapsed/20*12429/60:.1f} min estimated for full dataset")

for text, res in zip(sample[:3], results[:3]):
    print(text[:150])
    print(dict(zip(res["labels"], [round(s,2) for s in res["scores"]])))
    print()

9.1s for 20 rows -> ~94.6 min estimated for full dataset
I have a 2015 Chevy malibu with only 14,000 miles on it. It has an rebuilt title, has cold A/C, the interior is in fair conditions, everything is up t
{'a basic reliable daily commuter car': 0.97, 'a family-friendly car with practical space': 0.96, 'a fuel-efficient eco-friendly car': 0.07, 'an off-road or rugged utility vehicle': 0.04, 'a luxury car with premium features': 0.03, 'a sporty high-performance car': 0.0}

I have a 2002 Buick Lesabre with 176838 miles on it. It runs and am willing to take $1000 obo in the form of cash/Paypal/Venmo. It has newer tires on 
{'a family-friendly car with practical space': 0.8, 'a basic reliable daily commuter car': 0.77, 'an off-road or rugged utility vehicle': 0.34, 'a sporty high-performance car': 0.22, 'a luxury car with premium features': 0.17, 'a fuel-efficient eco-friendly car': 0.02}

BMW Series 3 328i year 2008 sunroof nice and clean interior and exterior no damadge price 7900.00
{

In [22]:
def build_classifier_input(row):
    return f"{row['year']} {row['manufacturer']} {row['model']}, a {row['type']} in {row['condition']} condition. {row['description_clean']}"

sample_rows = cars.sample(20, random_state=1)
texts = sample_rows.apply(build_classifier_input, axis=1).tolist()

start = time.time()
results = classifier(texts, vibe_labels, multi_label=True, hypothesis_template=hypothesis_template, batch_size=8)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 20 rows -> ~{elapsed/20*12429/60:.1f} min estimated for full dataset")

for text, res in zip(texts[:3], results[:3]):
    print(text[:150])
    print(dict(zip(res["labels"], [round(s,2) for s in res["scores"]])))
    print()

9.2s for 20 rows -> ~95.0 min estimated for full dataset
2015.0 chevrolet malibu, a unknown in good condition. I have a 2015 Chevy malibu with only 14,000 miles on it. It has an rebuilt title, has cold A/C, 
{'a family-friendly car with practical space': 0.96, 'a basic reliable daily commuter car': 0.96, 'an off-road or rugged utility vehicle': 0.37, 'a luxury car with premium features': 0.29, 'a fuel-efficient eco-friendly car': 0.22, 'a sporty high-performance car': 0.21}

2002.0 buick lesabre, a sedan in fair condition. I have a 2002 Buick Lesabre with 176838 miles on it. It runs and am willing to take $1000 obo in the 
{'a family-friendly car with practical space': 0.97, 'a basic reliable daily commuter car': 0.92, 'a luxury car with premium features': 0.41, 'a fuel-efficient eco-friendly car': 0.07, 'a sporty high-performance car': 0.02, 'an off-road or rugged utility vehicle': 0.01}

2008.0 bmw 328i, a sedan in excellent condition. BMW Series 3 328i year 2008 sunroof nice and cle

In [23]:
start = time.time()
results = classifier(
    texts, vibe_labels,
    multi_label=False,   # <- the change
    hypothesis_template=hypothesis_template,
    batch_size=8,
)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 20 rows -> ~{elapsed/20*12429/60:.1f} min estimated for full dataset")

for text, res in zip(texts[:3], results[:3]):
    print(text[:150])
    print(dict(zip(res["labels"], [round(s,2) for s in res["scores"]])))
    print()

8.8s for 20 rows -> ~90.8 min estimated for full dataset
2015.0 chevrolet malibu, a unknown in good condition. I have a 2015 Chevy malibu with only 14,000 miles on it. It has an rebuilt title, has cold A/C, 
{'a basic reliable daily commuter car': 0.41, 'a family-friendly car with practical space': 0.25, 'a luxury car with premium features': 0.13, 'an off-road or rugged utility vehicle': 0.09, 'a fuel-efficient eco-friendly car': 0.06, 'a sporty high-performance car': 0.06}

2002.0 buick lesabre, a sedan in fair condition. I have a 2002 Buick Lesabre with 176838 miles on it. It runs and am willing to take $1000 obo in the 
{'a basic reliable daily commuter car': 0.39, 'a family-friendly car with practical space': 0.39, 'a luxury car with premium features': 0.1, 'a fuel-efficient eco-friendly car': 0.05, 'a sporty high-performance car': 0.03, 'an off-road or rugged utility vehicle': 0.03}

2008.0 bmw 328i, a sedan in excellent condition. BMW Series 3 328i year 2008 sunroof nice and clea

In [24]:
import torch
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

True
True


In [25]:
import pandas as pd

cars = pd.read_csv("cars_with_vibes.csv")
vibe_cols = ["vibe_family", "vibe_sporty", "vibe_luxury", "vibe_ecofriendly", "vibe_offroad", "vibe_commuter"]

print(cars[vibe_cols].describe())
print()
print("dominant vibe distribution across all 12,429 rows:")
print(cars[vibe_cols].idxmax(axis=1).value_counts())

        vibe_family   vibe_sporty   vibe_luxury  vibe_ecofriendly  \
count  12429.000000  12429.000000  12429.000000      12429.000000   
mean       0.308032      0.098472      0.228716          0.090061   
std        0.240688      0.168220      0.253817          0.151068   
min        0.000358      0.000176      0.000478          0.000428   
25%        0.111718      0.009430      0.037739          0.015182   
50%        0.251177      0.028880      0.112948          0.036404   
75%        0.456892      0.097810      0.355330          0.082108   
max        0.996310      0.988340      0.989120          0.962717   

       vibe_offroad  vibe_commuter  
count  12429.000000   12429.000000  
mean       0.129431       0.145288  
std        0.199679       0.163015  
min        0.000324       0.000266  
25%        0.012711       0.025563  
50%        0.041561       0.086830  
75%        0.148497       0.206821  
max        0.989207       0.995460  

dominant vibe distribution across all 12,429

In [26]:
cars["tagged_description"].to_csv("tagged_description.txt", sep="\n", index=False, header=False)

In [27]:
from langchain_core.documents import Document

with open("tagged_description.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

documents = [Document(page_content=line) for line in lines]
print(len(documents))  # should be 12429, matching cars_with_vibes.csv row count

12429


In [28]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

db_cars = Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings(),
    persist_directory="chroma_db"
)
print("done")

done


In [29]:
result = db_cars.get(include=["embeddings"])
import numpy as np

embeddings = np.array(result["embeddings"], dtype=np.float32)
print(embeddings.shape)  # expect (12429, 1536)

np.save("car_embeddings.npy", embeddings)

(12429, 1536)


In [32]:
import numpy as np
import pandas as pd
from langchain_openai import OpenAIEmbeddings

query = "A car to teach children about nature".replace(" to teach children about nature", " for a family road trip")  # just a real test query
embedder = OpenAIEmbeddings()
query_vec = np.array(embedder.embed_query(query))

# cosine similarity, manually: dot product over norms
norms = np.linalg.norm(embeddings, axis=1)
query_norm = np.linalg.norm(query_vec)
sims = (embeddings @ query_vec) / (norms * query_norm)

top_idx = sims.argsort()[::-1][:5]

cars = pd.read_csv("cars_with_vibes.csv")
print(cars.iloc[top_idx][["year", "manufacturer", "model", "type", "description_clean"]])

         year manufacturer              model      type  \
11427  2016.0        dodge      grand caravan       van   
5088   2013.0        dodge  grand caravan sxt  mini-van   
8392   2017.0   volkswagen              jetta     sedan   
4578   2011.0        honda            odyssey  mini-van   
945    2015.0         ford          escape se       SUV   

                                       description_clean  
11427  Great Family vehicle, fits 7 passengers , fold...  
5088   Want the freedom of an RV, but the ability to ...  
8392   I love car but it doesn't fit our family. Our ...  
4578   Great family vehicle for you to take anywhere....  
945    Come check out this spacious rig that you can ...  


In [33]:
from search_engine import load_data, get_embedder, search_cars

cars, embeddings = load_data()
embedder = get_embedder()

results = search_cars(
    query="a reliable family SUV for road trips",
    cars=cars, embeddings=embeddings, embedder=embedder,
    filters={"fuel": "gas", "price_range": (5000, 30000)},
    vibe_label="Family",
    top_k=8
)
results[["year", "manufacturer", "model", "type", "price", "vibe_family", "score"]]

,year,manufacturer,model,type,price,vibe_family,score
11427,2016.0,dodge,grand caravan,van,12300,0.992804,0.896453
4329,2007.0,honda,odyssey ex-l,mini-van,5700,0.980127,0.882922
642,2008.0,chevrolet,trailblazer,SUV,5200,0.958736,0.882348
10688,2010.0,honda,odyssey,van,6500,0.996310,0.879191
4578,2011.0,honda,odyssey,mini-van,11500,0.994320,0.877786
4836,2017.0,kia,sedona lx,mini-van,16000,0.992492,0.875588
9642,2007.0,toyota,sienna,unknown,5800,0.976668,0.871675
11257,2006.0,toyota,sienna,van,5499,0.980002,0.871408


In [34]:
results2 = search_cars(
    query="a fun two-door car for weekend drives",
    cars=cars, embeddings=embeddings, embedder=embedder,
    vibe_label="Sporty",
    top_k=8
)
results2[["year", "manufacturer", "model", "type", "price", "vibe_sporty", "score"]]

,year,manufacturer,model,type,price,vibe_sporty,score
1324,2007.0,mazda,miata mx5,convertible,9700,0.938164,0.868654
2583,2007.0,ford,mustang saleen,coupe,37750,0.988252,0.853560
3022,1998.0,chevrolet,corvette,coupe,15983,0.943999,0.850741
2417,2000.0,chevrolet,camaro,coupe,3500,0.950669,0.849525
3873,2005.0,subaru,impreza wrx wagon,hatchback,14995,0.959081,0.847151
6047,2013.0,NaN,Mustang gt 2013,other,14499,0.956678,0.846473
3029,1998.0,chevrolet,camaro,coupe,8500,0.971988,0.846399
2834,2011.0,ford,mustang,coupe,12500,0.955309,0.844780


In [35]:
import pandas as pd

cars = pd.read_csv("cars_with_vibes.csv")
ev_mask = cars["description_clean"].str.contains(r"\btesla\b|\belectric\b|\bev\b", case=False, regex=True, na=False)
print("EV-keyword rows in current 12,429-row sample:", ev_mask.sum())

# also check the larger pre-sampling pool — our 1000-per-body-type cap wasn't stratified by fuel,
# so if EVs exist but are rare, random sampling could easily have excluded them by chance
cars_full = pd.read_csv("cars_cleaned.csv")
ev_mask_full = cars_full["description_clean"].str.contains(r"\btesla\b|\belectric\b|\bev\b", case=False, regex=True, na=False)
print("EV-keyword rows in full 108,390-row cleaned pool:", ev_mask_full.sum())
print(cars_full.loc[ev_mask_full, ["year", "manufacturer", "model", "fuel", "type"]].head(10))

EV-keyword rows in current 12,429-row sample: 213
EV-keyword rows in full 108,390-row cleaned pool: 213
        year manufacturer            model fuel type
145   2005.0          kia       sedona sel  gas  SUV
212   1994.0       toyota      4runner sr5  gas  SUV
334   2008.0        buick      enclave cxl  gas  SUV
475   2011.0    chevrolet         traverse  gas  SUV
542   2015.0       toyota      rav limited  gas  SUV
623   2016.0        acura              rdx  gas  SUV
670   2007.0        honda              cvr  gas  SUV
783   2015.0     cadillac  srx performance  gas  SUV
974   1996.0         jeep      cherokee xj  gas  SUV
1005  1997.0         ford  e350 super duty  gas  bus


In [36]:
from search_engine import load_data, get_embedder, search_cars

cars, embeddings = load_data()
embedder = get_embedder()

results = search_cars("electric car", cars, embeddings, embedder, top_k=8)
print(results[["year", "manufacturer", "model", "fuel", "similarity"]])

        year manufacturer    model      fuel  similarity
3484  2012.0       nissan  leaf sl  electric    0.869451
3294  2015.0       nissan     leaf  electric    0.860047
3419  2018.0       nissan     leaf  electric    0.858727
3710  2011.0       nissan  leaf sv  electric    0.856601
4154  2013.0       nissan   leaf s  electric    0.845845
5960  2013.0          NaN      gem  electric    0.843137
3335  2013.0       nissan     leaf  electric    0.840846
3393  2012.0       nissan     leaf  electric    0.840678


In [37]:
import pandas as pd
cars_full = pd.read_csv("cars_cleaned.csv")
cars_sample = pd.read_csv("cars_with_vibes.csv")

print("Full 108k pool, fuel counts:")
print(cars_full["fuel"].value_counts(dropna=False))
print()
print("Sampled 12,429, fuel counts:")
print(cars_sample["fuel"].value_counts(dropna=False))

Full 108k pool, fuel counts:
fuel
gas         11260
diesel        806
hybrid        175
other         113
NaN            38
electric       37
Name: count, dtype: int64

Sampled 12,429, fuel counts:
fuel
gas         11260
diesel        806
hybrid        175
other         113
NaN            38
electric       37
Name: count, dtype: int64


In [38]:
from search_engine import load_data, get_embedder, search_cars
import numpy as np

cars, embeddings = load_data()
embedder = get_embedder()

query = "family car sedan with good fuel mileage. I want automatic transmission. Electric car."
results = search_cars(query, cars, embeddings, embedder, top_k=10)
print(results[["year", "manufacturer", "model", "fuel", "transmission", "type", "similarity"]])

         year manufacturer                     model fuel transmission  \
5676   2011.0          NaN        smart coupe fortwo  gas    automatic   
12309  2009.0       subaru  legacy wagon outback ltd  gas    automatic   
3383   2013.0        mazda                         3  gas       manual   
8319   2012.0        honda                    accord  gas    automatic   
11521  2012.0       subaru     outback wagon premium  gas    automatic   
5966   2011.0          NaN        smart coupe fortwo  gas    automatic   
2389   2009.0      pontiac                        g5  gas       manual   
4225   2011.0       toyota                    sienna  gas    automatic   
3724   2017.0         ford                 fiesta se  gas       manual   
3998   2012.0       toyota                     prius  gas    automatic   

            type  similarity  
5676       other    0.835977  
12309      wagon    0.834521  
3383   hatchback    0.833965  
8319       sedan    0.833063  
11521      wagon    0.832811  

In [39]:
query_vec = np.array(embedder.embed_query(query))
norms = np.linalg.norm(embeddings, axis=1)
sims = (embeddings @ query_vec) / (norms * np.linalg.norm(query_vec) + 1e-8)

electric_mask = cars["fuel"] == "electric"
electric_sims = sims[electric_mask]
print("best electric similarity for this query:", electric_sims.max())
print("how many non-electric cars rank above the best electric match:", (sims > electric_sims.max()).sum(), "out of", len(cars))

best electric similarity for this query: 0.8206864307054039
how many non-electric cars rank above the best electric match: 37 out of 12429
